# Spark Structured API: DataFrames and SQL
Trong notebook trước, bạn đã thấy cách distributed processing sử dụng RDDs được thực hiện. Trong notebook này, chúng ta sẽ tìm hiểu về Spark's Structured API. Chúng ta sẽ xem cách bạn có thể sử dụng DataFrames và SQL để thực hiện các thao tác xử lý dữ liệu phổ biến. Đến cuối phần này, bạn sẽ có cảm nhận về điểm mạnh và điểm yếu của các phương pháp khác nhau này.

Sự khác biệt đầu tiên là Spark entrypoint. Đối với RDDs, entrypoint là 'SparkContext' (thường được đặt tên là sc). Còn đối với DataFrames, chúng ta sẽ sử dụng 'SparkSession', vốn mạnh mẽ hơn và dễ sử dụng hơn. Theo quy ước, chúng ta đặt tên SparkSession là spark, và tạo nó như sau:


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession \
    .builder \
    .getOrCreate()

24/08/21 08:15:51 WARN SparkContext: Another SparkContext is being constructed (or threw an exception in its constructor). This may indicate an error, since only one SparkContext should be running in this JVM (see SPARK-2243). The other SparkContext was created at:
org.apache.spark.api.java.JavaSparkContext.<init>(JavaSparkContext.scala:58)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance0(Native Method)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance(NativeConstructorAccessorImpl.java:62)
java.base/jdk.internal.reflect.DelegatingConstructorAccessorImpl.newInstance(DelegatingConstructorAccessorImpl.java:45)
java.base/java.lang.reflect.Constructor.newInstance(Constructor.java:490)
py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:247)
py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374)
py4j.Gateway.invoke(Gateway.java:238)
py4j.commands.ConstructorCommand.invokeConstructor(ConstructorCommand.java:80)
py4j.commands.Con

Chúng ta có thể sử dụng SparkSession để tạo DataFrames (như chúng ta sẽ thấy ngay sau đây) và các DataFrames này có thể được chuyển đổi thành RDDs. Tuy nhiên, nếu chúng ta muốn tạo RDDs trực tiếp thì phải thực hiện thông qua SparkContext. Một SparkContext được chứa trong SparkSession, và có thể được sử dụng như sau:

In [3]:
sc = spark.sparkContext
rdd = sc.parallelize(['a', 'b', 'c'])
rdd.collect()

['a', 'b', 'c']

## DataFrames from Python collections
Giống như chúng ta đã thấy với sc.parallelize cho RDDs, chúng ta có thể tạo một DataFrame từ một Python collection có sẵn. Bên cạnh chính collection đó, chúng ta cũng sẽ mô tả (một phần) cấu trúc dữ liệu bằng cách đặt tên cho các cột. Ngoài ra, chúng ta có thể chỉ định data types của các cột, nhưng trong trường hợp này chúng ta có thể để Spark tự động suy luận.

Đầu tiên, một list of tuples trong Python được tạo, gọi là phone_stock. Tiếp theo, chúng ta tạo một list gọi là columns chứa tên của tất cả các cột trong DataFrame. Sau đó, chúng ta sử dụng hai list này làm input cho hàm createDataFrame. Kết quả là DataFrame phone_df. Tiếp đó, chúng ta in ra type của cả phone_stock và phone_df.


In [4]:
phone_stock = [
    ('iPhone 6', 'Apple', 6, 549.00),
    ('iPhone 6s', 'Apple', 5, 585.00),
    ('iPhone 7', 'Apple', 11, 739.00),
    ('Pixel', 'Google', 8, 859.00),
    ('Pixel XL', 'Google', 2, 959.00),
    ('Galaxy S7', 'Samsung', 10, 539.00),
    ('Galaxy S6', 'Samsung', 5, 414.00),
    ('Galaxy A5', 'Samsung', 7, 297.00),
    ('Galaxy Note 7', 'Samsung', 0, 841.00)
]

columns = ['model', 'brand', 'stock', 'unit_price']

phone_df = spark.createDataFrame(phone_stock, columns)

print('the type of phoneStock: ' + str(type(phone_stock)))
print('the type of phone_df: ' + str(type(phone_df)))

the type of phoneStock: <class 'list'>
the type of phone_df: <class 'pyspark.sql.dataframe.DataFrame'>


Để xem một vài dòng của DataFrame, hãy sử dụng [`show()`](https://spark.apache.org/docs/latest/api/python/pyspark.sql.html#pyspark.sql.DataFrame.show). Mặc định, nó hiển thị 20 dòng, nhưng bạn có thể truyền vào số dòng mong muốn làm argument.

In [5]:
phone_df.show()

+-------------+-------+-----+----------+
|        model|  brand|stock|unit_price|
+-------------+-------+-----+----------+
|     iPhone 6|  Apple|    6|     549.0|
|    iPhone 6s|  Apple|    5|     585.0|
|     iPhone 7|  Apple|   11|     739.0|
|        Pixel| Google|    8|     859.0|
|     Pixel XL| Google|    2|     959.0|
|    Galaxy S7|Samsung|   10|     539.0|
|    Galaxy S6|Samsung|    5|     414.0|
|    Galaxy A5|Samsung|    7|     297.0|
|Galaxy Note 7|Samsung|    0|     841.0|
+-------------+-------+-----+----------+



In [6]:
phone_df.printSchema()

root
 |-- model: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- stock: long (nullable = true)
 |-- unit_price: double (nullable = true)



Giống như RDDs, chúng ta có một [`collect()`](https://spark.apache.org/docs/latest/api/python/pyspark.sql.html#pyspark.sql.DataFrame.collect)
 action, trả về toàn bộ dữ liệu từ một DataFrame về driver. Lưu ý rằng chúng ta nhận được các đối tượng `Row` chứa các cặp column name và value. Hãy nhớ rằng kết quả của `collect()` là một Python data structure (một list các đối tượng `Row`).

In [7]:
all_phones = phone_df.collect()
all_phones

[Row(model='iPhone 6', brand='Apple', stock=6, unit_price=549.0),
 Row(model='iPhone 6s', brand='Apple', stock=5, unit_price=585.0),
 Row(model='iPhone 7', brand='Apple', stock=11, unit_price=739.0),
 Row(model='Pixel', brand='Google', stock=8, unit_price=859.0),
 Row(model='Pixel XL', brand='Google', stock=2, unit_price=959.0),
 Row(model='Galaxy S7', brand='Samsung', stock=10, unit_price=539.0),
 Row(model='Galaxy S6', brand='Samsung', stock=5, unit_price=414.0),
 Row(model='Galaxy A5', brand='Samsung', stock=7, unit_price=297.0),
 Row(model='Galaxy Note 7', brand='Samsung', stock=0, unit_price=841.0)]

Làm việc trực tiếp với một list các đối tượng Row khá rườm rà. Để làm việc trực tiếp với dữ liệu ở phía driver, chúng ta thường chuyển đổi Spark DataFrame thành một pandas DataFrame. pandaslà một thư viện xử lý dữ liệu cho phép chúng ta thao tác với dữ liệu dạng bảng (tabular table). Nó phù hợp cho việc xử lý không quá nặng và dữ liệu không quá lớn để có thể chứa trong local memory (nếu không, tại sao chúng ta lại muốn dùng Spark?).

Spark DataFrames có một action toPandas(), cho phép lấy toàn bộ dữ liệu về driver và chuyển đổi nó thành một pandas DataFrame:

In [8]:
phone_df.toPandas()

,model,brand,stock,unit_price
0,iPhone 6,Apple,6,549.0
1,iPhone 6s,Apple,5,585.0
2,iPhone 7,Apple,11,739.0
3,Pixel,Google,8,859.0
4,Pixel XL,Google,2,959.0
5,Galaxy S7,Samsung,10,539.0
6,Galaxy S6,Samsung,5,414.0
7,Galaxy A5,Samsung,7,297.0
8,Galaxy Note 7,Samsung,0,841.0


Có một số cách để xem cấu trúc của một DataFrame: printSchema, schema và describe. printSchema đặc biệt hữu ích với các cấu trúc lồng nhau (nested structures) phức tạp, vì nó cung cấp một dạng dễ đọc cho con người (human-readable form):

In [9]:
phone_df.printSchema()

root
 |-- model: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- stock: long (nullable = true)
 |-- unit_price: double (nullable = true)



Lưu ý rằng tất cả các columns đều được liệt kê, kèm theo type của chúng và một giá trị boolean cho biết liệu giá trị của column đó có thể là NULL hay không.

Schema cũng có thể được liệt kê một cách programmatically. Bằng cách gọi schema, chúng ta sẽ thấy cấu trúc của DataFrame theo các Spark types. Có thể định nghĩa một schema trong code bằng cách sử dụng những types này, mặc dù ở đây chúng ta sẽ không làm điều đó.

In [10]:
phone_df.schema

StructType([StructField('model', StringType(), True), StructField('brand', StringType(), True), StructField('stock', LongType(), True), StructField('unit_price', DoubleType(), True)])

Chúng ta cũng có thể xem chi tiết hơn về cấu trúc của các fields, trong đó các columns được định nghĩa:

In [11]:
phone_df.schema.fields

[StructField('model', StringType(), True),
 StructField('brand', StringType(), True),
 StructField('stock', LongType(), True),
 StructField('unit_price', DoubleType(), True)]

[`describe`](https://spark.apache.org/docs/latest/api/python/pyspark.sql.html#pyspark.sql.DataFrame.describe) sẽ tính summary statistics cho các numeric columns và string columns:

In [12]:
phone_df.describe().show()

24/08/21 08:19:06 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+---------+-------+------------------+------------------+
|summary|    model|  brand|             stock|        unit_price|
+-------+---------+-------+------------------+------------------+
|  count|        9|      9|                 9|                 9|
|   mean|     null|   null|               6.0| 642.4444444444445|
| stddev|     null|   null|3.5355339059327378|220.82295573100586|
|    min|Galaxy A5|  Apple|                 0|             297.0|
|    max| iPhone 7|Samsung|                11|             959.0|
+-------+---------+-------+------------------+------------------+



## Data extraction


Bây giờ khi chúng ta đã có dữ liệu trong một DataFrame, chúng ta muốn sử dụng nó để thao tác dữ liệu. Hãy bắt đầu bằng cách chọn các subsets của dữ liệu: các specific columns và/hoặc rows.

### Selecting columns

Thường thì chúng ta không quan tâm đến tất cả các **columns** của dữ liệu. **DataFrames** giúp việc chọn một **subset** rất dễ dàng bằng cách sử dụng phương thức **[`select`](https://spark.apache.org/docs/latest/api/python/pyspark.sql.html#pyspark.sql.DataFrame.describe)**. Hãy lưu ý rằng chúng ta không sửa đổi **DataFrame** gốc, mà đang tạo ra một **DataFrame** mới.



In [13]:
# Select only the model column
model_df = phone_df.select("model")
model_df.show()

+-------------+
|        model|
+-------------+
|     iPhone 6|
|    iPhone 6s|
|     iPhone 7|
|        Pixel|
|     Pixel XL|
|    Galaxy S7|
|    Galaxy S6|
|    Galaxy A5|
|Galaxy Note 7|
+-------------+



Chúng ta cũng có thể **rename** một **column** bằng cách sử dụng
 [`expr`](https://spark.apache.org/docs/2.2.0/api/python/pyspark.sql.html#pyspark.sql.functions.expr).

In [14]:
from pyspark.sql.functions import expr
mymodel_df = phone_df.select("brand", expr("model as mymodel"))
mymodel_df.show()

+-------+-------------+
|  brand|      mymodel|
+-------+-------------+
|  Apple|     iPhone 6|
|  Apple|    iPhone 6s|
|  Apple|     iPhone 7|
| Google|        Pixel|
| Google|     Pixel XL|
|Samsung|    Galaxy S7|
|Samsung|    Galaxy S6|
|Samsung|    Galaxy A5|
|Samsung|Galaxy Note 7|
+-------+-------------+



In [15]:
# Select both the brand and model columns
bm_df = phone_df.select('brand', 'model')
bm_df.show()

+-------+-------------+
|  brand|        model|
+-------+-------------+
|  Apple|     iPhone 6|
|  Apple|    iPhone 6s|
|  Apple|     iPhone 7|
| Google|        Pixel|
| Google|     Pixel XL|
|Samsung|    Galaxy S7|
|Samsung|    Galaxy S6|
|Samsung|    Galaxy A5|
|Samsung|Galaxy Note 7|
+-------+-------------+



## Bài tập 1
chọn cột `model` và `stock` từ `phone_df`:

In [ ]:
# TODO: Replace <FILL IN> with appropriate code
# Select the model and stock columns
ms_df = phone_df.<FILL_IN>
ms_df.show()

### Filtering rows
Chúng ta có thể **filter** các **rows** cụ thể bằng cách sử dụng phương thức **DataFrame [`filter`](https://spark.apache.org/docs/latest/api/python/pyspark.sql.html#pyspark.sql.DataFrame.filter)**. Lưu ý rằng phương thức **[`where`](https://spark.apache.org/docs/latest/api/python/pyspark.sql.html#pyspark.sql.DataFrame.where)** là một **alias** của **`filter`**. Cách chỉ định **columns** giống như khi sử dụng phương thức **select**:


In [16]:
# Select rows with phones from Google
google_df = phone_df.<FILL_IN>

google_df.show()

+--------+------+-----+----------+
|   model| brand|stock|unit_price|
+--------+------+-----+----------+
|   Pixel|Google|    8|     859.0|
|Pixel XL|Google|    2|     959.0|
+--------+------+-----+----------+



## Bài tập 2
chọn các hàng với `unit_price` nhỏ hơn 550.00

In [ ]:
# TODO: Replace <FILL IN> with appropriate code

cheap_df = phone_df.filter(<FILL IN>)
cheap_df.show()

In [ ]:
# TODO: Liệt kê các điện thoại có stock > 5.

## Bài tập 3
Thêm cột total_value = stock * unit_price

In [ ]:
# TODO: Thêm cột total_value = stock * unit_price

Nhiều điều kiện **filter** có thể được chỉ định bằng cách sử dụng **[boolean operations](https://docs.python.org/3/library/stdtypes.html#boolean-operations-and-or-not)** của **Python**:


In [ ]:
phone_df.filter((phone_df.brand == 'Apple') | (phone_df.brand == 'Google')).show()

### Ordering rows

chúng ta sử dụng phương thức [`orderBy`](https://spark.apache.org/docs/latest/api/python/pyspark.sql.html#pyspark.sql.DataFrame.orderBy) để sắp xếp dữ liệu:

In [ ]:
phone_df.orderBy('unit_price').show()

#### Note: Columns specifications

Trong các ví dụ trước, chúng ta đã sử dụng nhiều loại ***column specifications*** khác nhau để **select** và **filter** dữ liệu. Đôi khi các cách viết phức tạp hơn là cần thiết, vì các phiên bản ngắn gọn có thể gây mơ hồ cho **Spark's parser**. Ví dụ, tất cả các cách dưới đây là tương đương:

```
bm_df = phone_df.select("brand", "model")
bm_df = phone_df.select(["brand", "model"])
bm_df = phone_df.select(phone_df["brand"], phone_df["model"])
```

Trong ô tiếp theo, chúng ta sử dụng một chuỗi các phương thức của **DataFrame** rất giống với ngôn ngữ truy vấn **SQL** được dùng cho một số cơ sở dữ liệu.
Lưu ý rằng chúng ta chỉ sử dụng tên của các **columns**. Ngoài ra, hãy chú ý đến việc sử dụng **double quotes** và **single quotes** trong phương thức **[`where`](https://spark.apache.org/docs/latest/api/python/pyspark.sql.html#pyspark.sql.DataFrame.where)**.


In [ ]:
phone_df.select("model", "unit_price").where("brand='Apple'").orderBy('stock', ascending=False).show()

Một cách khác để làm giống như ô phía trên là sử dụng **`phone_df["brand"]`** trong **where clause**. Cách này dài hơn để gõ nhưng trực quan hơn, rõ ràng và dễ đọc hơn. Với cú pháp này, **Spark parser** sẽ không gặp sự mơ hồ.


In [ ]:
phone_df.select("model", "unit_price").where(phone_df["brand"]=="Apple").orderBy('stock', ascending=False).show()

## Assignment 3
Select all phones with a unit price larger than 300 and of which there are more than two in stock. Display the remaining phones, ordered by brand, followed by stock. Use whatever column specification syntax you prefer.

In [ ]:
<FILL IN>

## Aggregating data
An important part of data processing is the ability to combine multiple records, like we did with `reduceByKey`. In the DataFrame API this is a two-step process:

First you group the data using the `groupBy` method. `groupBy` can operate on one or multiple columns. It will not actually perform the grouping but create a reference to a `GroupedData` object:

In [ ]:
grouped_df = phone_df.groupBy('brand')
print(type(grouped_df))

After the data is grouped we can apply one of the standard aggregation functions on it. They are listed at the [GroupedData](https://spark.apache.org/docs/latest/api/python/pyspark.sql.html#pyspark.sql.GroupedData) API documentation. These are: `min`, `max`, `mean`, `sum` and `count`. We can apply an aggregation to all columns or to a subset of the columns.

In [ ]:
# Minimum for all columns
min_df = grouped_df.min('unit_price')

min_df.toPandas()

Notice that the `min(unit_price)` is the name of the new column. If you want to rename a column use [`withColumnRenamed`](https://spark.apache.org/docs/latest/api/python/pyspark.sql.html#pyspark.sql.DataFrame.withColumnRenamed). As arguments this method takes the old name and new name of the column.

## Assignment 4

Compute the maximum  of the unit_price per brand and rename the resulting column to `max`.
(We assume you can do this in one line. Feel free to adapt the cell and use more lines if you want.)

In [ ]:
# TODO: Replace <FILL IN> with appropriate code
max_df = <FILL IN>
max_df.toPandas()

Finally, we can combine different aggregations per column using the [`agg`](https://spark.apache.org/docs/latest/api/python/pyspark.sql.html#pyspark.sql.GroupedData.agg) method on a GroupedData instance:

In [ ]:
# Take the sum of the stock column, and calculate the mean of the unit_price column, in one go
sum_df = grouped_df.agg({'stock': 'sum', 'unit_price': 'mean'})

sum_df.show()

## SQL
The SQL API aims to be ANSI-SQL SQL2003 and Hive-SQL compatible. The expressiveness is very similar to the DataFrame API. You can access the SQL API from the SparkSession by using `spark.sql`. Below is a query performed using Spark's DataFrame API:

In [ ]:
# DataFrame version
res_df = phone_df.filter(phone_df['stock'] > 7).select('model')
res_df.show()

The SQL version of the query requires us to 'register' the DataFrame as an SQL table: 

In [ ]:
# SQL version

# Register the phone_df DataFrame within SQL as a table with name 'phones'
phone_df.createOrReplaceTempView('phones')

# Perform the SQL query on the 'phones' table
res_df = spark.sql('SELECT model FROM phones WHERE stock > 7')
res_df.show()

## Joining with other data sets
Often you want to combine multiple datasets on a shared column. In this example we create an extra table with information about the phone manufacturer:

In [ ]:
companies = [
    ('Google', 'USA', 1998, 'Sundar Pichai'),
    ('Samsung', 'South Korea', 1938 ,'Oh-Hyun Kwon' ),
    ('Apple', 'USA', 1976 ,'Tim Cook')
]

columns = ['company_name', 'hq_country', 'founding_year', 'ceo']

company_df = spark.createDataFrame(companies, columns)
company_df.show()

To join two DataFrames, we use the `join` method on one of the DataFrames. This method takes two arguments: (1) the other DataFrame, and (2) a join relation. Here we join the two DataFrames on the brand/company_name columns:

In [ ]:
joined_df = phone_df.join(company_df, phone_df['brand'] == company_df['company_name'])
joined_df.show()

Here is an example of a more complicated query that combines multiple steps:

In [ ]:
# All the models from USA companies with more than 7 items in stock
result = phone_df \
    .join(company_df, phone_df['brand'] == company_df['company_name']) \
    .filter(company_df['hq_country'] == 'USA') \
    .filter(phone_df['stock'] > 7) \
    .select('model')

result.show()

## Assignment 5

The problem below was taken from Coursera's MOOC [Big Data Analysis with Scala and Spark](https://www.coursera.org/learn/scala-spark-big-data) by the École Polytechnique Fédérale de Lausanne. We adapted the problem for PySpark.

Let's assume we have a dataset with posts from a discussion forum. The entries of the dataset consist of an authorID, the name of a subforum, the number of likes and a date. The data frame is constructed in the following cell.

**We would like to know how many likes each author posted on each subforum. The table should show per subforum how many likes each author has, the highest number of likes first.**

In [ ]:
from  pyspark.sql import Row
from pyspark.sql.functions import count


posts = [{'authorID' : 4, 'subforum': 'java', 'likes': 5, 'date' : 'sept 5'},
         {'authorID' : 1, 'subforum': 'python', 'likes': 3, 'date' : 'sept 4'},
        {'authorID' : 2, 'subforum': 'python', 'likes': 35, 'date' : 'sept 3'},
        {'authorID' : 3, 'subforum': 'java', 'likes': 1, 'date' : 'sept 5'},
        {'authorID' : 4, 'subforum': 'java', 'likes': 14, 'date' : 'sept 5'},
        {'authorID' : 3, 'subforum': 'python', 'likes': 12, 'date' : 'sept 3'},
        {'authorID' : 3, 'subforum': 'java', 'likes': 14, 'date' : 'sept 5'},
        {'authorID' : 3, 'subforum': 'java', 'likes': 10, 'date' : 'sept 5'},
        {'authorID' : 2, 'subforum': 'python', 'likes': 21, 'date' : 'sept 5'}]

rdd = spark.sparkContext.parallelize(posts)
df_posts = spark.createDataFrame(rdd.map(lambda x : Row(**x)))

Please use a [groupBy](https://spark.apache.org/docs/latest/api/python/pyspark.sql.html#pyspark.sql.DataFrame.groupBy), the [sum aggregation function](https://spark.apache.org/docs/latest/api/python/pyspark.sql.html#pyspark.sql.GroupedData.sum) and an [orderBy](https://spark.apache.org/docs/latest/api/python/pyspark.sql.html#pyspark.sql.DataFrame.orderBy) to come up with the desired dataFrame. Note that you want to order in descending order.
Also note, that you can use [`groupBy`](https://spark.apache.org/docs/latest/api/python/pyspark.sql.html#pyspark.sql.DataFrame.groupBy) and `orderBy` on more than one column.

If you get confused, break the problem into steps.

In [ ]:
<FILL IN>

## Conversion to/from RDD

Sometimes you want to do data manipulations which would be very easy with RDD operations, but complicated with the DataFrame API. Fortunately you can convert between DataFrames and RDDs of type 'Row'. Going from DataFrame to RDD is quite simple. Going back from RDD to DataFrame is more difficult because you need to re-apply the schema.

In [ ]:
phone_rdd = phone_df.rdd
plural_rdd = phone_rdd.map(lambda r: r.brand + 's')
plural_rdd.collect()

# Reading structured files/sources
One of the advantages of DataFrames is the ability to read already structured data and automatically import the structure in Spark. Spark contains readers for a number of formats such as csv, json, parquet, orc, text and jdbc. There are also third-party readers/connectors for databases such as MongoDB and Cassandra.

Here we read the json-formatted tweets that we also used in the last notebook. As you can see the complicated JSON schema is inferred.

In [ ]:
tweet_df = spark.read.format("json").load('../data/tweets.json')
tweet_df.printSchema()

This structure is squeezed into a table. When we convert to Pandas we can see what the first tweet looks like in a DataFrame.

In [ ]:
tweet_df.toPandas().head(1)

## Assignment 6
Select the name and screen_name of the user, the text field and the lang field.

**Hint**: nested fields can be selected using the dot notation, i.e. `df.select('<parent>.<child>')`.

In [ ]:
name_df = tweet_df.<FILL IN>
name_df.toPandas().head(15)

## Assignment 7
Count the number of tweets per user, and display the top 10 most-tweeting users.

In [ ]:
<FILL IN>

## Word count in DataFrames

It is also possible to use DataFrames for less-structured data such as text. Here we show how you could do word count with DataFrames.

The following chained query contains a number of methods you haven't seen before, and we'll go through it line by line.

In [ ]:
from pyspark.sql.functions import explode, split

spark \
    .read.text('../data/shakespeare.txt') \
    .select(explode(split("value", "\W+")).alias("word")) \
    .groupBy("word") \
    .count() \
    .orderBy("count", ascending=0).show()

To see what happens here, we break it down into steps. First we read in the data file and inspect the DataFrame. It contains one column, called `value` by default.

In [ ]:
swan_df = spark.read.text('../data/shakespeare.txt')
swan_df.show()

The column name `value` explains why it is mentioned inside the `split` function. Let's call the `select` method but omit `explode` and see what happens. Notice, that with `alias` we rename the column.

In [ ]:
split_df = swan_df.select(split("value", "\W+").alias("word"))
split_df.show()

Looking at the schema, we can see that `word` is actually an array of strings:

In [ ]:
split_df.printSchema()

Instead, we would like to have a row for each word, which is where [`explode`](http://spark.apache.org/docs/latest/api/python/pyspark.sql.html#pyspark.sql.functions.explode) comes in. It has a similar meaning as `flatMap` in Spark RDDs. It gets rid of lists:

In [ ]:
swan_df.select(explode(split("value", "\W+")).alias("word")).show()

### User-defined functions

In the previous example we used the built-in split function. It is also possible to define and use a custom user-defined function, or UDF. We'll show an example for the phone stock DataFrame first:

In [ ]:
from pyspark.sql.types import StringType
from pyspark.sql.functions import udf

exp_udf = udf(lambda price: "Expensive" if price >= 500 else "Inexpensive", StringType())

phone_df.withColumn("cost", exp_udf(phone_df['unit_price'])).show()

In this manner, we can apply specialized function, like tokenizers, on DataFrames. However, we first must register them as UDFs and cannot simply define them inline with lambda functions like we can with RDDs.

Below we define a very simple tokenizer, just as an example. It uses Python's string `split`, and also lowers the case of the text.

In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.types import ArrayType, StringType

def my_tokenize(s):
    s = s.lower()
    words = s.split()
    return words

returnType = ArrayType(StringType())

tokenize_udf = udf(my_tokenize, returnType)

## Assignment 8
Use the `my_tokenize` function from the last cell to count words on the Shakespeare DataFrame `swan_df` instead of usng the `split` function. Display the top 10 most occurring words.

In [ ]:
<FILL IN>